# quantls — Architecture & Usage Guide

This notebook documents the full system: how data flows through the pipeline,
what each module is responsible for, and how to run a backtest or generate
live trading signals.

---

## Table of Contents

1. [System overview](#1-system-overview)
2. [Configuration](#2-configuration)
3. [Data pipeline](#3-data-pipeline)
4. [Signal generation](#4-signal-generation)
5. [Portfolio optimizer](#5-portfolio-optimizer)
6. [Backtesting](#6-backtesting)
7. [Live trading](#7-live-trading)
8. [Reporting](#8-reporting)
9. [End-to-end cheatsheet](#9-end-to-end-cheatsheet)

---
## 1 System overview

```
┌───────────���───────────────────────���─────────────────────────────────────┐
│                          quantls data flow                              │
│                                                                         │
│  config.py                                                              │
│  └─ Config  ──────────────────────────────────────────────────────┐     │
│                                                                   │     │
│  data/                                                            ▼     │
│  ├─ universe.py  → S&P 500 tickers (Wikipedia)                         │
│  └─ market.py    → yfinance OHLCV (used by pipeline internally)        │
│                                                                         │
│  pipeline/  ──────────────────── writes to ──────► FeatureStore (SQLite)│
│  ├─ Stage 1 price.py       close, volume                               │
│  ├─ Stage 2 fundamentals.py  ebit, roe, roa, sgr …  (Polygon API)     │
│  ├─ Stage 3 scores.py       pe, pb, growth_score …  (derived)         │
│  └─ Stage 4 sentiment.py    bull_minus_bear         (Polygon + FinBERT)│
│                                      │                                  │
│                                      ▼                                  │
│  signals/                     FeatureStore.get()                        │
│  ├─ factors.py   → 6 cross-sectional factor z-scores + combined        │
│  └─ predictor.py → Lasso regression → ML scores                        │
│            │                                                            │
│            └──── combined = factors[combined] + ml_zscored * ml_weight │
│                                      │                                  │
│                                      ▼                                  │
│  portfolio/optimizer.py  cvxpy LP → target weights                     │
│                                      │                                  │
│               ┌──────────────────────┴────────────────────┐            │
│               ▼                                           ▼            │
│  engine/backtest.py                          engine/live.py            │
│  daily P&L loop                              retrain() + generate()    │
│               │                                           │            │
│               ▼                                           ▼            │
│  reporting/summary.py                        broker/IBBroker.rebalance │
└─────────────────────────────────────────────────────────────────────────┘
```

### Module responsibilities at a glance

| Module | Responsibility |
|---|---|
| `config.py` | Single `Config` dataclass — all tunable parameters |
| `data/universe.py` | S&P 500 ticker list from Wikipedia |
| `data/market.py` | Bulk yfinance OHLCV download |
| `data/fundamentals.py` | Polygon quarterly filings → CSV cache |
| `pipeline/price.py` | Stage 1 — OHLCV → SQLite `prices` table |
| `pipeline/fundamentals.py` | Stage 2 — filings forward-filled daily → `fundamentals` table |
| `pipeline/scores.py` | Stage 3 — derived ratios (P/E, P/B, ROE…) → `scores` table |
| `pipeline/sentiment.py` | Stage 4 — FinBERT on Polygon news → `sentiment` table |
| `pipeline/store.py` | `FeatureStore` — unified read/write to SQLite |
| `signals/factors.py` | 6 cross-sectional factor z-scores + weighted `combined` |
| `signals/predictor.py` | Lasso regression on lagged price + fundamental features |
| `portfolio/optimizer.py` | cvxpy LP — maximize alpha, dollar-neutral, leverage cap |
| `engine/backtest.py` | Historical simulation loop |
| `engine/live.py` | Production: `retrain()` then `generate_signals()` |
| `reporting/summary.py` | Sharpe, Sortino, Calmar, drawdown, win rate |

---
## 2 Configuration

Everything is driven by a single `Config` dataclass. Instantiate it with
defaults and override only what you need.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../../..'))  # project root

from src.quantls.config import Config
import dataclasses, pprint

cfg = Config()
pprint.pprint(dataclasses.asdict(cfg))

### Key parameters explained

#### Universe & positions
```python
universe_size  = 100   # top-N S&P 500 tickers to consider
total_positions = 40   # 20 longs + 20 shorts (equal split)
max_gross_leverage = 1.0  # Σ|w| ≤ 1.0 → dollar-neutral book
```

#### ML model
```python
holding_days   = 1     # forward return horizon the Lasso predicts
lookback_days  = 126   # training window (~6 months of trading days)
lasso_alpha    = 0.01  # L1 regularisation strength
ml_weight      = 1.5   # scale on z-scored ML scores before blending
ml_retrain_days = 21   # minimum calendar days between retrains (≈monthly)
```

#### Factor weights  
These must sum to 1.0. Each weight is the fraction of the combined signal
contributed by that factor before the ML score is blended in.
```python
factor_weight_value       = 0.20  # EBIT/EV (or 21d return proxy)
factor_weight_quality     = 0.20  # ROE (or rolling vol proxy)
factor_weight_sentiment   = 0.10  # FinBERT bull-minus-bear 3d MA
factor_weight_growth      = 0.15  # revenue/EPS growth score
factor_weight_value_score = 0.15  # composite value (P/E, P/B, yield)
factor_weight_style_score = 0.20  # 50/50 blend of value + growth
```

#### Data sources
```python
polygon_api_key = ''   # leave empty → price-only mode (no fundamentals/sentiment)
db_path = '.cache/features.db'  # SQLite feature store location
```

> **Price-only mode**: when `polygon_api_key` is empty the pipeline skips
> Stages 2–4 and uses price/volume proxies for every fundamental factor.
> Results are still valid but less informative.

---
## 3 Data pipeline

The pipeline is run once per date range and caches every result to SQLite.
Subsequent runs skip stages that already have complete data.

### Stage 1 — Prices (`pipeline/price.py`)

Downloads adjusted OHLCV via `yfinance` and writes to the `prices` table.
Tickers with fewer than 80% of expected trading days are silently dropped.

```
prices table schema
  date TEXT, ticker TEXT, open REAL, close REAL, volume REAL
  PRIMARY KEY (date, ticker)
```

### Stage 2 — Fundamentals (`pipeline/fundamentals.py`)

Fetches quarterly filings from Polygon `/vX/reference/financials`.  
Each filing is forward-filled to a daily business-day calendar so every
trading day has the most recent point-in-time value (no look-ahead bias).

Fields stored: `ebit, net_income, equity, assets, eps, equity_per_share,
shares, long_term_debt, cash, revenue, revenue_growth,
equity_per_share_growth, sustainable_growth_rate`

Rate limit: 5 req/min (free tier). Set `polygon_requests_per_minute` higher
on paid tiers.

### Stage 3 — Scores (`pipeline/scores.py`)

Reads `prices` + `fundamentals` and computes derived metrics daily:

| Score | Formula |
|---|---|
| `roe` | net_income / equity |
| `roa` | net_income / assets |
| `pe_ratio` | close / eps |
| `pb_ratio` | close / equity_per_share |
| `earning_yield` | eps / close |
| `ebit_to_ev` | ebit / (mkt_cap + debt − cash) |
| `growth_score` | cross-sectional z-score of revenue_growth, eps_growth, sgr |
| `value_score` | cross-sectional z-score of earning_yield, −pb, −pe |
| `style_score` | 50% value_score + 50% growth_score |

### Stage 4 — Sentiment (`pipeline/sentiment.py`)

For each `(ticker, date)` pair:  
1. Fetches up to 50 news articles from Polygon  
2. Scores each headline+description with **FinBERT** (P(positive) − P(negative))  
3. Averages scores → `bull_minus_bear` ∈ [−1, +1]  
4. Applies 3-day rolling mean → `bull_minus_bear_3d`

FinBERT is lazy-loaded on first use (~1 GB model download on first run).

In [ ]:
# Running the pipeline manually (the engines do this automatically)
from src.quantls.config import Config
from src.quantls.data import get_sp500_tickers
from src.quantls.pipeline import PipelineRunner

cfg = Config(
    universe_size=10,           # small universe for demo
    start_date='2023-01-01',
    end_date='2023-06-30',
    polygon_api_key='',         # price-only mode
    db_path='.cache/demo.db',
)

tickers = get_sp500_tickers()[:cfg.universe_size]
print('Universe:', tickers)

runner = PipelineRunner(cfg)
store  = runner.run(tickers, cfg.start_date, cfg.end_date)
print('Pipeline complete. Store type:', type(store))

In [ ]:
# Inspecting the feature store
import pandas as pd

# Point-in-time snapshot for a single date
snapshot = store.get(pd.Timestamp('2023-03-15'), tickers)
print('Snapshot shape:', snapshot.shape)
snapshot.head()

In [ ]:
# Reading a full date range from one table
prices = store.get_range('2023-01-01', '2023-03-31', tickers, 'prices')
print('Prices shape:', prices.shape)
prices.head()

---
## 4 Signal generation

Signals are generated at each rebalance date from two sources that are
blended into a single `combined` score per ticker.

### 4.1 Factor scores (`signals/factors.py`)

Each factor is winsorized at the 5th/95th percentile and then z-scored
cross-sectionally (removes the cross-sectional mean and scales by std).

```
combined = Σ weight_i × factor_i
         = 0.20×value + 0.20×quality + 0.10×sentiment
           + 0.15×growth + 0.15×value_score + 0.20×style_score
```

When Polygon data is unavailable, each factor falls back to a price/volume proxy:

| Factor | With Polygon | Price-only proxy |
|---|---|---|
| `value` | EBIT/EV | 21-day return |
| `quality` | ROE | −rolling vol (21d) |
| `sentiment` | FinBERT 3d MA | −5-day return |
| `growth` | growth_score | 252-day return |
| `value_score` | scores.value_score | 63-day return |
| `style_score` | scores.style_score | volume ratio (21d/63d) |

### 4.2 ML predictor (`signals/predictor.py`)

A **Lasso regression** is trained on a stacked `(date × ticker, feature)` matrix
and predicts the `holding_days`-forward return for each ticker.

**Feature set** (12 features when fundamentals are available):
- Price features: 1d return, 5d return, 21d return, volume ratio (21d/63d)
- Fundamental features: earning_yield, pb_ratio, pe_ratio, roa,
  sustainable_growth_rate, equity_per_share_growth, growth_score, value_score

**Retrain frequency**: the model skips retraining if fewer than
`ml_retrain_days` (default 21) calendar days have elapsed since the last fit.
This avoids expensive weekly refits during backtesting while still adapting
roughly monthly.

### 4.3 Blending

```
ml_zscored = z-score(ml_scores)
combined   = factors['combined'] + ml_zscored × ml_weight
```

The final `combined` series is the alpha signal passed to the optimizer.

In [ ]:
# Computing signals manually
from src.quantls.signals import compute_factors, Predictor
from src.quantls.engine.backtest import _load_prices
import pandas as pd

close, volume = _load_prices(store, tickers, cfg.start_date, cfg.end_date)

as_of = pd.Timestamp('2023-03-31')

# Take the lookback window ending on as_of
loc   = close.index.get_loc(as_of)
c_win = close.iloc[max(0, loc - cfg.lookback_days + 1): loc + 1]
v_win = volume.iloc[max(0, loc - cfg.lookback_days + 1): loc + 1]

features = store.get(as_of, tickers)
factors  = compute_factors(c_win, v_win, cfg, features=features)
print('Factor columns:', factors.columns.tolist())
factors[['combined']].sort_values('combined', ascending=False)

In [ ]:
# Adding ML scores
predictor = Predictor(cfg)
ml_scores = predictor.fit_predict(c_win, v_win, store=store, as_of=as_of)

ml_z     = (ml_scores - ml_scores.mean()) / (ml_scores.std() + 1e-9)
combined = factors['combined'].add(ml_z * cfg.ml_weight, fill_value=0)

print('Top 5 longs:')
print(combined.nlargest(5))
print('\nTop 5 shorts:')
print(combined.nsmallest(5))

---
## 5 Portfolio optimizer

`portfolio/optimizer.py` wraps a **cvxpy linear programme** that allocates
the 40 selected tickers (top-20 and bottom-20 by `combined`) into weights.

### LP formulation

```
maximise    αᵀ w

subject to  Σ wᵢ = 0                    (dollar-neutral)
            Σ|wᵢ| ≤ max_gross_leverage   (gross exposure cap, default 1.0)
            −max_pos ≤ wᵢ ≤ max_pos      (concentration limit)

where  max_pos = 2 / total_positions = 0.05  (5% per name at 40 positions)
```

The solver cascade is OSQP → SCS → ECOS; zero weights are returned only
if all three fail.

In [ ]:
from src.quantls.portfolio import optimize_portfolio

universe = (
    combined.nlargest(cfg.long_n).index
    .union(combined.nsmallest(cfg.short_n).index)
)

weights = optimize_portfolio(combined[universe], cfg)

print(f'Longs: {(weights > 0).sum()}  Shorts: {(weights < 0).sum()}')
print(f'Gross leverage: {weights.abs().sum():.4f}')
print(f'Net exposure:   {weights.sum():.6f}')
weights.sort_values(ascending=False)

---
## 6 Backtesting

`engine/backtest.py` runs an event-driven simulation over historical data.

### What happens on each day

```
for each trading day:
    mark portfolio to market  (daily P&L)
    if day is a rebalance date (default: every Friday):
        compute factors + ML scores for the lookback window
        select top-N long / short universe
        run LP optimizer → new target weights
        (position changes are assumed to fill at close, no transaction costs)
```

### Key design points

- **Point-in-time correctness**: the feature store uses filing dates as keys,
  so quarterly fundamentals only become visible on the day they were filed.
- **No transaction costs or slippage** are modelled (a known limitation).
- The pipeline runs once before the loop and all features are cached; the
  loop itself only does linear algebra — no network calls.

In [ ]:
# Minimal backtest
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

from src.quantls.config import Config
from src.quantls.engine import Backtest

cfg = Config(
    universe_size=50,
    total_positions=20,
    start_date='2022-01-01',
    end_date='2023-12-31',
    polygon_api_key='',      # price-only mode — no API key needed
    db_path='.cache/bt.db',
)

bt = Backtest(cfg)
results = bt.run()           # returns a DataFrame of daily portfolio values
results.tail()

In [ ]:
# Print performance summary
from src.quantls.reporting.summary import print_summary, compute_metrics

print_summary(results, cfg)

# Or get the metrics dict for programmatic use
metrics = compute_metrics(results, cfg)
metrics

In [ ]:
# Plot equity curve
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Equity curve
results['portfolio_value'].plot(ax=axes[0], color='steelblue', linewidth=1.5)
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title('Equity Curve')
axes[0].grid(True, alpha=0.3)

# Drawdown
pv = results['portfolio_value']
drawdown = (pv / pv.cummax() - 1) * 100
drawdown.plot(ax=axes[1], color='firebrick', linewidth=1, fill=True, alpha=0.4)
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_title('Drawdown')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Parameter sensitivity: vary lasso_alpha
import pandas as pd

rows = []
for alpha in [0.001, 0.01, 0.05, 0.1]:
    c = Config(
        universe_size=50, total_positions=20,
        start_date='2022-01-01', end_date='2023-12-31',
        polygon_api_key='', db_path='.cache/bt.db',
        lasso_alpha=alpha,
    )
    r = Backtest(c).run()
    m = compute_metrics(r, c)
    m['lasso_alpha'] = alpha
    rows.append(m)

pd.DataFrame(rows).set_index('lasso_alpha')

---
## 7 Live trading

`engine/live.py` splits the production flow into two separate steps that
should be scheduled independently:

```
Friday 4:15 pm  →  LiveEngine.retrain()           # fit Lasso, save model
Friday 9:31 am  →  LiveEngine.generate_signals()  # load model, return weights
                →  IBBroker.rebalance(weights)     # submit orders
```

### Why separate retrain from signal generation?

- **retrain** uses after-market closing prices (full Friday bar available).
- **generate_signals** runs at market open using that saved model — no
  retraining latency on the execution path.
- If the model file already exists and you only need signals, you can skip
  retraining entirely.

### Connection requirements

Live trading requires a running **TWS** or **IB Gateway** instance:

| Session | Port |
|---|---|
| TWS paper trading | 7497 |
| TWS live trading | 7496 |
| IB Gateway paper | 4002 |
| IB Gateway live | 4001 |

In [ ]:
# Step 1 — retrain (run Friday after close)
from src.quantls.config import Config
from src.quantls.engine import LiveEngine

cfg = Config(
    universe_size=100,
    total_positions=40,
    polygon_api_key='YOUR_POLYGON_KEY',   # required for live mode
    db_path='.cache/live.db',
)

engine = LiveEngine(cfg)

# Fits the Lasso on the most recent lookback window and saves to
# .cache/predictor.joblib  (path is next to db_path)
engine.retrain()

In [ ]:
# Step 2 — generate signals (run Friday at market open)
weights = engine.generate_signals()
print(f'Longs: {(weights > 0).sum()}  Shorts: {(weights < 0).sum()}')
weights.sort_values(ascending=False)

In [ ]:
# Step 3 — submit to Interactive Brokers
# Always test with dry_run=True first.
from src.broker import IBBroker

with IBBroker(
    host='127.0.0.1',
    port=7497,          # paper trading port
    client_id=1,
    order_type='LMT',   # limit orders at mid-price
    stagger_seconds=0.5,
    dry_run=True,       # set False to submit real orders
) as broker:
    broker.rebalance(weights)

### Live trading sequence diagram

```
Friday 4:15 pm
    LiveEngine.retrain()
        └── _prepare()
                ├── get_sp500_tickers()
                ├── PipelineRunner.run()  ←── updates SQLite cache
                └── _load_prices(store)   ←── reads from store (no re-download)
        └── Predictor.train(close, volume, store)
        └── Predictor.save('.cache/predictor.joblib')

Friday 9:31 am
    LiveEngine.generate_signals()
        └── _prepare()  (same as above — pipeline skips cached stages)
        └── Predictor.load('.cache/predictor.joblib')
        └── Predictor.infer(close, volume, store)
        └── compute_factors(close, volume, cfg, features)
        └── combined = factors + ml_z × ml_weight
        └── optimize_portfolio(combined[top40], cfg)
        └── return weights

    IBBroker.rebalance(weights)
        └── _get_equity()          ←── TWS/Gateway API
        └── _get_positions()
        └── _get_prices(tickers)
        └── compute delta shares
        └── sells first, then buys  (avoids margin spikes)
```

---
## 8 Reporting

`reporting/summary.py` exposes two functions:

- `print_summary(results, cfg)` — formatted console output
- `compute_metrics(results, cfg)` — returns a dict for programmatic use

### Metrics reference

| Metric | Formula | What it tells you |
|---|---|---|
| Total Return | (final / initial − 1) × 100 | Absolute P&L over the period |
| Ann. Volatility | daily_std × √252 × 100 | Annualised risk |
| Sharpe Ratio | mean_daily / std_daily × √252 | Return per unit of total risk |
| Sortino Ratio | mean_daily / downside_std × √252 | Return per unit of downside risk only |
| Max Drawdown | min(pv / cummax(pv) − 1) × 100 | Worst peak-to-trough loss |
| Calmar Ratio | total_return / \|max_drawdown\| | Return relative to worst loss |
| Win Rate | (days > 0) / total_days × 100 | % of days with positive P&L |

In [ ]:
# Example: compare two configs side by side
import pandas as pd
from src.quantls.config import Config
from src.quantls.engine import Backtest
from src.quantls.reporting.summary import compute_metrics

shared = dict(
    universe_size=50, total_positions=20,
    start_date='2022-01-01', end_date='2023-12-31',
    polygon_api_key='', db_path='.cache/bt.db',
)

configs = {
    'default':      Config(**shared),
    'hi_leverage':  Config(**shared, max_gross_leverage=1.5),
    'no_ml':        Config(**shared, ml_weight=0.0),
}

rows = {}
for name, c in configs.items():
    r = Backtest(c).run()
    rows[name] = compute_metrics(r, c)

pd.DataFrame(rows).T

---
## 9 End-to-end cheatsheet

### Quickstart — backtest (no API key needed)

```python
from src.quantls.config import Config
from src.quantls.engine import Backtest
from src.quantls.reporting.summary import print_summary

cfg     = Config(start_date='2022-01-01', end_date='2023-12-31')
results = Backtest(cfg).run()
print_summary(results, cfg)
```

### Quickstart — live trading (requires Polygon key + TWS)

```python
from src.quantls.config import Config
from src.quantls.engine import LiveEngine
from src.broker import IBBroker

cfg    = Config(polygon_api_key='YOUR_KEY')
engine = LiveEngine(cfg)

# Run once Friday after close
engine.retrain()

# Run Friday morning
weights = engine.generate_signals()

with IBBroker(port=7497, dry_run=False) as broker:
    broker.rebalance(weights)
```

### Common gotchas

| Symptom | Likely cause | Fix |
|---|---|---|
| `RuntimeError: Price data missing` | Pipeline hasn't run or wrong `db_path` | Check `db_path` in Config matches between pipeline and engine |
| All factor scores are proxies | `polygon_api_key` is empty | Set the key or accept proxy mode |
| Slow first sentiment run | FinBERT (~1 GB) downloading | Normal, cached after first run |
| `FileNotFoundError: No saved model` | `generate_signals` called before `retrain` | Run `engine.retrain()` first |
| Lasso all zeros | `lasso_alpha` too high | Lower `lasso_alpha` (try 0.001) |
| `ConnectionRefused` from IBBroker | TWS/Gateway not running | Start TWS or IB Gateway and enable API in settings |
| Optimizer returns zero weights | LP infeasible (tiny universe) | Increase `universe_size` or decrease `total_positions` |

### File layout

```
src/quantls/
├── config.py              ← start here
├── data/
│   ├── universe.py        ← S&P 500 tickers
│   ├── market.py          ← yfinance wrapper
│   ├── fundamentals.py    ← Polygon filings (CSV cache)
│   └── sentiment.py       ← Polygon news fetcher
├── pipeline/
│   ├── runner.py          ← orchestrates stages 1-4
│   ├── store.py           ← SQLite read/write
│   ├── price.py           ← stage 1
│   ├── fundamentals.py    ← stage 2
│   ├── scores.py          ← stage 3
│   └── sentiment.py       ← stage 4
├── signals/
│   ├── factors.py         ← cross-sectional z-scores
│   └── predictor.py       ← Lasso ML model
├── portfolio/
│   └── optimizer.py       ← cvxpy LP
├── engine/
│   ├── backtest.py        ← historical simulation
│   └── live.py            ← production signal generator
└── reporting/
    └── summary.py         ← performance metrics
```